In [1]:
import pandas as pd
from openai import AsyncOpenAI, OpenAI
import asyncio
import json
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

deepseek_key = os.getenv("DEEPSEEK_API_KEY")

In [ ]:

client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")
print(client.models.list())

SyncPage[Model](data=[Model(id='deepseek-chat', created=None, object='model', owned_by='deepseek'), Model(id='deepseek-reasoner', created=None, object='model', owned_by='deepseek')], object='list')


In [4]:
import sqlite3

In [ ]:
def dump_csv(base_path):
    db_path = os.path.join(base_path, "local_snapshot.db")
    output_folder = os.path.join(base_path, "csv")

    os.makedirs(output_folder, exist_ok=True)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    
    for table_name_tuple in tables:
        table_name = table_name_tuple[0]

        
        safe_table_name = f'"{table_name}"'

        print(f"Обработка таблицы: {table_name}")

        
        query = f'SELECT * FROM {safe_table_name};'
        df = pd.read_sql_query(query, conn)

        
        
        safe_filename = table_name.replace(" ", "_").replace("/", "_").replace("\\", "_")
        csv_file_path = os.path.join(output_folder, f"{safe_filename}.csv")

        
        df.to_csv(csv_file_path, index=False)

    conn.close()

In [6]:
with open("tables-qa/set/sakila/1/QA.txt") as f:
    question = f.read().split("\nA:")[0].strip("Q:")

question

'Какой email у клиента с номером 1, если известно, что у него есть связанный адрес и город?'

In [7]:
def get_full_context_prompt(full_context):
    return f"""
    Ответь строго по данным из таблиц: 
    дай краткий и точный ответ на вопрос, 
    используя только информацию из таблиц. 
    Верни результат в формате JSON без пояснений, 
    комментариев или дополнительного текста.    

    Формат вывода: 
    {{"answer": 27}}

    {full_context}
    """

In [8]:
from tqdm import tqdm

In [9]:
results = []

In [10]:
results_df = pd.DataFrame(results)

In [ ]:
import json
import os
import asyncio
from tqdm.asyncio import tqdm_asyncio
from openai import AsyncOpenAI
import pandas as pd
from typing import List, Dict, Any


client = AsyncOpenAI(
    api_key=deepseek_key,
    base_url="https://api.deepseek.com"
)

async def ask_async(prompt, **kwargs):
    """Асинхронный метод для запросов"""
    response = await client.chat.completions.create(
        model=kwargs.get("model", "deepseek-reasoner"),
        messages=[{"role": "user", "content": prompt}],
        temperature=kwargs.get("temperature", 0.3),
        extra_body={"thinking": {"type": "enabled"}} if kwargs.get("think", True) else None,
        response_format={"type": "json_object"} if kwargs.get("json_output", True) else None
    )
    return {
        "content": response.choices[0].message.content,
        "thinking": getattr(response.choices[0].message.reasoning_content, 'thinking', None)
    }

async def process_item(db_name: str, number: str, base_dir: str, results_df: pd.DataFrame):
    """Обработка одного элемента асинхронно"""
    
    if not results_df.empty and ((results_df["db_name"] == db_name) & (results_df["number"] == number)).any():
        print(f"Skip {db_name} {number}")
        return None
    
    item_path = os.path.join(base_dir, db_name, number, "full_context.md")
    
    try:
        with open(item_path, 'r') as f:
            full_context = f.read()
    except FileNotFoundError:
        return {
            "db_name": db_name, 
            "number": number, 
            "error": f"File not found: {item_path}",
            "response": None
        }
    
    try:
        
        prompt = get_full_context_prompt(full_context)
        response = await ask_async(
            prompt,
            think=True,
            json_output=False,
            model="deepseek-reasoner"
        )
        
        res_dict = {"db_name": db_name, "number": number, "response": response}
        
        
        try:
            res_json = json.loads(response["content"])
            if "answer" in res_json:
                res_dict["answer"] = res_json["answer"]
        except json.JSONDecodeError:
            
            pass
        except Exception as e:
            res_dict["parse_error"] = str(e)
        
        return res_dict
        
    except Exception as e:
        return {
            "db_name": db_name, 
            "number": number, 
            "error": f"API exception: {str(e)}",
            "response": None
        }

async def process_batch(items: List[tuple], max_concurrent: int = 10):
    """Обработка батча с ограничением на одновременные запросы"""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(db_name, number):
        async with semaphore:
            return await process_item(db_name, number, base_dir, results_df)
    
    
    tasks = [process_with_semaphore(db_name, number) for db_name, number in items]
    
    
    results = []
    for task in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
        result = await task
        if result:
            results.append(result)
            print(result)
            print(f"Processed: {result.get('db_name')} {result.get('number')}")
    
    return results

async def main():
    base_dir = "tables-qa/set"
    results = []
    
    
    results_file = "results.csv"
    try:
        results_df = pd.read_csv(results_file)
        print(f"Загружено {len(results_df)} существующих результатов")
    except FileNotFoundError:
        results_df = pd.DataFrame(columns=["db_name", "number"])
        print("Существующие результаты не найдены, начинаем с нуля")
    
    
    items_to_process = []
    for db_name in os.listdir(base_dir):
        db_dir = os.path.join(base_dir, db_name)
        if not os.path.isdir(db_dir):
            continue
            
        for number in os.listdir(db_dir):
            
            number_path = os.path.join(db_dir, number)
            if not os.path.isdir(number_path):
                continue
                
            
            context_file = os.path.join(number_path, "full_context.md")
            if not os.path.exists(context_file):
                print(f"Warning: No full_context.md in {db_name}/{number}")
                continue
            
            
            if ((results_df["db_name"] == db_name) & (results_df["number"] == int(number))).any():
                print(f"Skip {db_name} {number}")
                continue
            
            items_to_process.append((db_name, number))
    
    print(f"Найдено {len(items_to_process)} элементов для обработки")
    
    if not items_to_process:
        print("Нет элементов для обработки")
        return
    
    
    batch_size = 10
    for i in range(0, len(items_to_process), batch_size):
        batch = items_to_process[i:i+batch_size]
        batch_num = i//batch_size + 1
        total_batches = (len(items_to_process) + batch_size - 1)//batch_size
        
        print(f"\nОбрабатываю батч {batch_num}/{total_batches} ({len(batch)} элементов)")
        
        batch_results = await process_batch(batch, max_concurrent=10)
        valid_results = [r for r in batch_results if r is not None]
        results.extend(valid_results)
        
        
        if valid_results:
            new_df = pd.DataFrame(valid_results)
            
            
            batch_file = f"results_batch_{batch_num}.csv"
            new_df.to_csv(batch_file, index=False)
            print(f"Сохранен батч {batch_num} в {batch_file}")
            
            
            results_df = pd.concat([results_df, new_df], ignore_index=True)
        
        
        if i + batch_size < len(items_to_process):
            print("Пауза между батчами...")
            await asyncio.sleep(2)
    
    
    if results:
        final_df = pd.DataFrame(results)
        final_df.to_csv("final_results.csv", index=False)
        print(f"\nОбработка завершена! Сохранено {len(results)} результатов")
    else:
        print("\nНет новых результатов для сохранения")

In [18]:
base_dir = "tables-qa/set"
results_df = pd.DataFrame()

await main()

Загружено 83 существующих результатов
Skip pubs 10
Skip pubs 5
Skip pubs 2
Skip pubs 3
Skip pubs 4
Skip pubs 1
Skip pubs 9
Skip pubs 7
Skip pubs 6
Skip pubs 8
Skip northwind 10
Skip northwind 5
Skip northwind 2
Skip northwind 3
Skip northwind 4
Skip northwind 1
Skip northwind 9
Skip northwind 7
Skip northwind 6
Skip northwind 8
Skip NCAA 10
Skip NCAA 5
Skip NCAA 2
Skip NCAA 3
Skip NCAA 4
Skip NCAA 1
Skip NCAA 9
Skip NCAA 7
Skip NCAA 6
Skip NCAA 8
Skip lahman_2014 10
Skip lahman_2014 5
Skip lahman_2014 2
Skip lahman_2014 3
Skip lahman_2014 4
Skip lahman_2014 1
Skip lahman_2014 9
Skip lahman_2014 7
Skip lahman_2014 6
Skip lahman_2014 8
Skip sakila 10
Skip sakila 5
Skip sakila 2
Skip sakila 3
Skip sakila 4
Skip sakila 1
Skip sakila 9
Skip sakila 7
Skip sakila 6
Skip sakila 8
Skip Hockey 10
Skip Hockey 5
Skip Hockey 2
Skip Hockey 3
Skip Hockey 4
Skip Hockey 1
Skip Hockey 9
Skip Hockey 7
Skip Hockey 6
Skip Hockey 8
Skip ErgastF1 10
Skip ErgastF1 5
Skip ErgastF1 2
Skip ErgastF1 3
Skip Ergast

 10%|█         | 1/10 [00:02<00:18,  2.03s/it]

{'db_name': 'Basketball_men', 'number': '2', 'error': 'API exception: Error code: 400 - {\'error\': {\'message\': "This model\'s maximum context length is 131072 tokens. However, you requested 291080 tokens (291080 in the messages, 0 in the completion). Please reduce the length of the messages or completion.", \'type\': \'invalid_request_error\', \'param\': None, \'code\': \'invalid_request_error\'}}', 'response': None}
Processed: Basketball_men 2


 20%|██        | 2/10 [00:07<00:32,  4.07s/it]

{'db_name': 'Mondial', 'number': '1', 'response': {'content': '{"answer": 402946.25}', 'thinking': None}, 'answer': 402946.25}
Processed: Mondial 1


...

{'db_name': 'Basketball_men', 'number': '4', 'response': {'content': '{"answer": 1970}', 'thinking': None}, 'answer': 1970}
Processed: Basketball_men 4


 86%|████████▌ | 6/7 [00:23<00:04,  4.17s/it]

{'db_name': 'Basketball_men', 'number': '9', 'response': {'content': '{"answer": "West"}', 'thinking': None}, 'answer': 'West'}
Processed: Basketball_men 9


100%|██████████| 7/7 [00:45<00:00,  6.54s/it]

{'db_name': 'Basketball_men', 'number': '8', 'response': {'content': '{"answer": 2034}', 'thinking': None}, 'answer': 2034}
Processed: Basketball_men 8
Сохранен батч 2 в results_batch_2.csv

Обработка завершена! Сохранено 17 результатов


In [2]:
import pandas as pd

In [ ]:
df = pd.read_csv("results.csv")

In [6]:
len(df)

83

In [7]:
df = pd.concat([df, pd.read_csv("final_results.csv")], ignore_index=False)

len(df)

100

In [8]:
df.to_csv("full_context_results.csv")

In [17]:
df.to_csv("results.csv")

In [37]:
pd.DataFrame(results).to_csv("results.csv")

In [28]:
for db_name in tqdm(os.listdir(base_dir)):
    db_dir = os.path.join(base_dir, db_name)
    for number in tqdm(os.listdir(db_dir), leave=False):
        if ((results_df["db_name"] == db_name) & (results_df["number"] == number)).any():
            print(db_name, number)

  0%|          | 0/10 [00:00<?, ?it/s]

pubs 10
pubs 5
pubs 2
pubs 3
pubs 4
pubs 1
pubs 9
pubs 7
pubs 6
pubs 8


100%|██████████| 10/10 [00:00<00:00, 325.09it/s]


In [55]:
pd.DataFrame(results).head()

,db_name,number,response,answer
0,pubs,10,"{\n ""answer"": ""801 826-0752""\n}",801 826-0752
1,pubs,5,"{\n ""answer"": 0\n}",0
2,pubs,2,"{\n ""answer"": 10.98\n}",10.98
3,pubs,3,"{\n ""answer"": 0\n}",0
4,pubs,4,"{\n ""answer"": 12.0\n}",12.0


In [21]:
ask_sync("Какая самая богатая страна в Африке?", json_output=False)

{'content': 'Самой богатой страной Африки по **общему объему ВВП** является **Нигерия** (по данным МВФ и Всемирного банка). Её экономика, крупнейшая на континенте, основана главным образом на нефтегазовом секторе.\n\nОднако если измерять богатство по **ВВП на душу населения** (показатель благосостояния среднего жителя), то самой богатой страной Африки являются **Сейшельские Острова**.\n\nЭто важное различие:\n\n### 1. По общему номинальному ВВП (размер экономики):\n   *   **Нигерия** (~ $390 млрд) — самая большая экономика.\n   *   **Египет** (~ $380 млрд) — очень близко.\n   *   **ЮАР** (~ $370 млрд) — на третьем месте.\n\nЭти три страны традиционно являются экономическими тяжеловесами континента.\n\n### 2. По ВВП на душу населения (уровень жизни):\n   *   **Сейшельские Острова** (~ $20,000) — лидер благодаря туризму и офшорным финансовым услугам при небольшом населении.\n   *   **Маврикий** (~ $19,500) — диверсифицированная экономика с развитым финансовым сектором.\n   *   **Габон** 

In [43]:
import ast
import json

In [51]:
base_dir = "tables-qa/set"

for db_name in os.listdir(base_dir):
    db_dir = os.path.join(base_dir, db_name)
    for number in os.listdir(db_dir):
        with open(os.path.join(db_dir, number, "QA.txt")) as f:
            text = f.read()
            question = text.split("\nA:")[0].strip("Q:")
            answer = text.split("\nA:")[1].strip("\nA:")
        print(answer)
        if "(" in answer:
            evaluated_list = ast.literal_eval(answer)
            string_list = [str(item) for item in evaluated_list]
            answer = " ".join(string_list)

        print(answer)
        with open(os.path.join(db_dir, number, "QA.json"), "w") as f:
            json.dump({"question": question, "answer": answer}, f, ensure_ascii=False)

('219 547-9982',)
219 547-9982
(283.8125,)
283.8125
(7.192,)
7.192
(937.8874999999999,)
937.8874999999999
(63.1578947368421,)
63.1578947368421
(468.94374999999997,)
468.94374999999997
('Cheryl',)
Cheryl
(66.66666666666667,)
66.66666666666667
(11.95,)
11.95
(1875.7749999999999,)
1875.7749999999999
(19.2,)
19.2
(696.0,)
696.0
(78,)
78
(40,)
40
(22,)
22
(73,)
73
(50,)
50
(4,)
4
(3,)
3
(851.2,)
851.2
('Florida St',)
Florida St
(24.857142857142858,)
24.857142857142858
(20.0,)
20.0
(17.0,)
17.0
(0.5,)
0.5
('Canisius',)
Canisius
('Clemson',)
Clemson
(76.56923076923077,)
76.56923076923077
(15.67922874671341,)
15.67922874671341
(17.4,)
17.4
('Ted', 'Williams')
Ted Williams
(46,)
46
(15360000,)
15360000
(3,)
3
(244,)
244
(65,)
65
('Joe', 'Nathan')
Joe Nathan
(6,)
6
(94.9090909090909,)
94.9090909090909
(347.0,)
347.0
('RUSSELL', 'BRINSON')
RUSSELL BRINSON
(8.98,)
8.98
(66.89,)
66.89
(10.99,)
10.99
(10.99,)
10.99
('MARY.SMITH@sakilacustomer.org',)
MARY.SMITH@sakilacustomer.org
('PATRICIA', 'JOHNSO

In [47]:
import ollama

In [ ]:
response = ollama.chat(
    model='your_model_name',
    messages=[{'role': 'user', 'content': 'Привет!'}]
)
print(response['message']['content'])

In [ ]:
base_dir = "tables-qa/set"

for db_name in tqdm(os.listdir(base_dir)):
    db_dir = os.path.join(base_dir, db_name)
    for number in os.listdir(db_dir):
        with open(os.path.join(db_dir, number, "QA.json")) as f:
            text = json.load(f)
            question = text["question"]
            answer = text["answer"]
        
        translated_question = ollama.chat(
            model='zongwei/gemma3-translator:4b',
            messages=[{'role': 'user', 'content': f"""Translate this question from Russian to English. Return only translation, with no additional text
                       Question: {question}"""}]
        )['message']['content']
        

        
        with open(os.path.join(db_dir, number, "QA_english.json"), "w") as f:
            json.dump({"question": translated_question, "answer": answer}, f, ensure_ascii=False)

100%|██████████| 10/10 [00:52<00:00,  5.25s/it]


In [71]:
def get_sql_to_text_prompt(query):
    return f"""
    Convert the following SQL query into a natural-language question in English, 
    as if it were asked by someone unfamiliar with databases or SQL. 
    The question should be easily understandable to a layperson, must not include terms like "table", 
    "JOIN", or "query", and should accurately convey the meaning of the SQL operations (filtering, combining data, 
    aggregation, etc.) using everyday language. Phrase it either as an interrogative sentence 
    (starting with words like "How", "What", "How many", etc.) or as a request ("Show me", "Find").

    Return only the formulated question, with no additional text. Answer should be in English
    SQL query:  
    {query}
    """

In [73]:
from mistralai import Mistral

In [75]:
def get_mistral_response(messages, model="mistral-large-latest", api_key=None, stream=False):
    """
    Функция для обращения к Mistral API
    
    Args:
        messages (list): Список сообщений в формате [{"role": "user", "content": "..."}, ...]
        model (str): Название модели (по умолчанию "mistral-small-latest")
        api_key (str): API ключ (если не указан, берется из переменной окружения MISTRAL_API_KEY)
        stream (bool): Флаг потоковой передачи (по умолчанию False)
    
    Returns:
        object: Ответ от Mistral API
    """
    if api_key is None:
        api_key = os.getenv("MISTRAL_API_KEY", "")
    
    with Mistral(api_key=api_key) as client:
        response = client.chat.complete(
            model=model,
            messages=messages,
            stream=stream
        )
    
    return response

In [81]:
from dotenv import load_dotenv
load_dotenv()

True

In [83]:
mistral_key = os.getenv("MISTRAL_API_KEY")

In [ ]:
base_dir = "tables-qa/set"

for db_name in tqdm(os.listdir(base_dir)):
    db_dir = os.path.join(base_dir, db_name)
    for number in tqdm(os.listdir(db_dir)):
        with open(os.path.join(db_dir, number, "QA.json")) as f:
            text = json.load(f)
            
            answer = text["answer"]
        try:
            with open(os.path.join(db_dir, number, "aggregation_query.sql")) as f:
                sql = f.read()
        except Exception:
            with open(os.path.join(db_dir, number, "query.sql")) as f:
                sql = f.read()
        
        translated_question = get_mistral_response(
            messages=[{'role': 'user', 'content': get_sql_to_text_prompt(sql)}], api_key=mistral_key
        ).choices[0].message.content
        
        print(sql, translated_question)

        
        with open(os.path.join(db_dir, number, "QA_english.json"), "w") as f:
            json.dump({"question": translated_question, "answer": answer}, f, ensure_ascii=False)
    
    print(db_name)